# Regresión Logística - Clasificación de Sistema Operativo

## Actividad 2 - Clase 4: Aprendizaje Supervisado

El objetivo de esta actividad es predecir qué **sistema operativo** utiliza un usuario (Windows, Macintosh o Linux) según los datos de comportamiento registrados en una web mediante Google Analytics.

### Variables de entrada (features):
- **duracion**: Duración de la visita en segundos
- **paginas**: Cantidad de páginas vistas durante la sesión
- **acciones**: Cantidad de acciones del usuario (click, scroll, checkbox, sliders, etc.)
- **valor**: Suma del valor de las acciones (cada acción tiene una valoración de importancia)

### Variable de salida (target):
- **0** → Windows
- **1** → Macintosh
- **2** → Linux


## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente ✓")


## 2. Carga y exploración del dataset

In [ ]:
# Cargamos el dataset
df = pd.read_csv('usuarios_win_mac_lin.csv')

print("=== Primeras filas del dataset ===")
print(df.head(10))
print(f"\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")


In [ ]:
print("=== Información general del dataset ===")
print(df.info())
print("\n=== Estadísticas descriptivas ===")
print(df.describe())


In [ ]:
print("=== Distribución de clases (sistema operativo) ===")
conteo = df['clase'].value_counts().sort_index()
etiquetas = {0: 'Windows', 1: 'Macintosh', 2: 'Linux'}
for clase, cantidad in conteo.items():
    print(f"  Clase {clase} ({etiquetas[clase]}): {cantidad} registros")


## 3. Visualización de los datos

In [ ]:
# Mapeamos las clases a nombres legibles
df['sistema'] = df['clase'].map({0: 'Windows', 1: 'Macintosh', 2: 'Linux'})

# Colores para cada clase
colores = {0: '#0078D7', 1: '#999999', 2: '#F7A30C'}
color_puntos = df['clase'].map(colores)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de usuarios por Sistema Operativo', fontsize=14, fontweight='bold')

# Gráfico 1: Duración vs Valor coloreado por clase
axes[0].scatter(df['duracion'], df['valor'], c=color_puntos, alpha=0.7, edgecolors='k', linewidths=0.4, s=60)
axes[0].set_xlabel('Duración (segundos)')
axes[0].set_ylabel('Valor de las acciones')
axes[0].set_title('Duración vs Valor')

# Gráfico 2: Páginas vs Acciones coloreado por clase
axes[1].scatter(df['paginas'], df['acciones'], c=color_puntos, alpha=0.7, edgecolors='k', linewidths=0.4, s=60)
axes[1].set_xlabel('Páginas vistas')
axes[1].set_ylabel('Cantidad de acciones')
axes[1].set_title('Páginas vs Acciones')

# Leyenda
leyenda = [
    mpatches.Patch(color='#0078D7', label='Windows (0)'),
    mpatches.Patch(color='#999999', label='Macintosh (1)'),
    mpatches.Patch(color='#F7A30C', label='Linux (2)')
]
for ax in axes:
    ax.legend(handles=leyenda, loc='upper right')

plt.tight_layout()
plt.savefig('visualizacion_datos.png', dpi=100, bbox_inches='tight')
plt.show()
print("Gráfico guardado como 'visualizacion_datos.png'")


## 4. Preparación de los datos

In [ ]:
# Separamos las features (X) del target (y)
X = df[['duracion', 'paginas', 'acciones', 'valor']].values
y = df['clase'].values

print(f"Features (X): {X.shape}  →  {X.shape[0]} muestras, {X.shape[1]} atributos")
print(f"Target  (y): {y.shape}  →  valores posibles: {np.unique(y)}")


In [ ]:
# Dividimos en conjunto de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de prueba:        {X_test.shape[0]} muestras")


## 5. Creación y entrenamiento del modelo de Regresión Logística

In [ ]:
# Creamos el clasificador de Regresión Logística
clasificador = LogisticRegression(max_iter=1000, random_state=42)

# Entrenamos el modelo con los datos de entrenamiento
clasificador.fit(X_train, y_train)

print("Modelo entrenado correctamente ✓")
print(f"Clases detectadas: {clasificador.classes_}  →  {[etiquetas[c] for c in clasificador.classes_]}")


## 6. Predicción

In [ ]:
# Realizamos predicciones sobre el conjunto de prueba
y_pred = clasificador.predict(X_test)

print("=== Primeras 15 predicciones vs valores reales ===")
print(f"{'Real':<12} {'Predicho':<12} {'Correcto'}")
print("-" * 35)
for real, pred in zip(y_test[:15], y_pred[:15]):
    ok = "✓" if real == pred else "✗"
    print(f"{etiquetas[real]:<12} {etiquetas[pred]:<12} {ok}")


## 7. Evaluación del modelo

In [ ]:
# Calculamos la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión (Accuracy): {accuracy:.2%}")
print()

# Reporte de clasificación completo
print("=== Reporte de Clasificación ===")
print(classification_report(
    y_test, y_pred,
    target_names=['Windows (0)', 'Macintosh (1)', 'Linux (2)']
))


In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
nombres_clases = ['Windows\n(0)', 'Macintosh\n(1)', 'Linux\n(2)']

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set(
    xticks=np.arange(cm.shape[1]),
    yticks=np.arange(cm.shape[0]),
    xticklabels=nombres_clases,
    yticklabels=nombres_clases,
    title='Matriz de Confusión',
    ylabel='Valor Real',
    xlabel='Valor Predicho'
)

# Agregar los valores dentro de la matriz
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=100, bbox_inches='tight')
plt.show()
print("Matriz guardada como 'matriz_confusion.png'")


## 8. Predicción con nuevos datos

Ahora probamos el modelo con datos de un usuario nuevo, que no estaba en el dataset original.


In [ ]:
# Ejemplo de un usuario nuevo
# [duracion_seg, paginas_vistas, cantidad_acciones, valor_acciones]
nuevo_usuario = np.array([[300, 3, 12, 36]])

prediccion = clasificador.predict(nuevo_usuario)
probabilidades = clasificador.predict_proba(nuevo_usuario)

print("=== Predicción para un usuario nuevo ===")
print(f"  Datos del usuario: duración={nuevo_usuario[0][0]}s, páginas={nuevo_usuario[0][1]}, acciones={nuevo_usuario[0][2]}, valor={nuevo_usuario[0][3]}")
print()
print(f"  Sistema operativo predicho: {etiquetas[prediccion[0]]} (clase {prediccion[0]})")
print()
print("  Probabilidades por clase:")
for clase, prob in zip(clasificador.classes_, probabilidades[0]):
    print(f"    {etiquetas[clase]:>12}: {prob:.2%}")


## 9. Conclusiones

En esta actividad aplicamos **Regresión Logística** para clasificar el sistema operativo de usuarios de un sitio web en base a su comportamiento de navegación.

### Resultados obtenidos:
- El modelo fue entrenado con el **80%** de los datos y evaluado con el **20%** restante.
- La **Regresión Logística** es un modelo lineal diseñado para problemas de clasificación discreta.
- El modelo aprende a separar las clases (Windows, Macintosh, Linux) trazando fronteras de decisión lineales en el espacio de las features.

### Observaciones:
- Las variables **acciones** y **valor** parecen tener mayor poder discriminativo entre los sistemas operativos.
- Los usuarios de **Macintosh** tienden a registrar más acciones y mayor valor por visita.
- Los usuarios de **Linux** tienden a tener sesiones más cortas con menos páginas visitadas.

### Diferencia con Regresión Lineal:
| | Regresión Lineal | Regresión Logística |
|---|---|---|
| Salida | Valor continuo | Clase discreta |
| Uso | Predecir números | Clasificar categorías |
| Función | Lineal | Función logística (sigmoide) |
| Ejemplo | Predecir precio | Clasificar SO del usuario |
